## Load modules

In [1]:
!module load scipy-stack/2023b
!module list

Lmod Warning: 
-------------------------------------------------------------------------------
The following dependent module(s) are not currently loaded: ipykernel/2026a
(required by: ipython-kernel/3.11)
-------------------------------------------------------------------------------




Currently Loaded Modules:
  1) CCconfig                 16) calibre/8.6.0
  2) gentoo/2023         (S)  17) libreqda/1.1.0
  3) gcccore/.12.3       (H)  18) flexiblascore/.3.3.1         (H)
  4) gcc/12.3            (t)  19) java/17 -> java/17.0.6       (t)
  5) hwloc/2.9.1              20) r/4.4.0                      (t)
  6) ucx/1.14.1               21) rstudio-server/4.4           (t)
  7) libfabric/1.18.0         22) python/3.11 -> python/3.11.5 (t)
  8) pmix/4.2.4               23) ipython-kernel/3.11          (S)
  9) ucc/1.2.0                24) openrefine/3.9.3
 10) openmpi/4.1.5       (m)  25) mlflow/3.8.1
 11) flexiblas/3.3.1          26) tensorboard/2.20.0
 12) blis/0.9.0               27) 

In [2]:
import numpy as np
import pandas as pd
import math as math
import matplotlib.pyplot as plt
import subprocess
import os
import scipy.stats as stats
import seaborn as sns
from collections import Counter
from tqdm import tqdm
import glob

## Remove Dropouts from Data

In [3]:
frequencies_df = pd.read_csv('../1.QC_and_Barcoding/frequencies.csv', sep=',', index_col=0)
frequencies_df

count_col_list = [
    'SC5314_TP0_YPD_1', 'SC5314_TP0_YPD_2', 'SC5314_TP0_YPD_3', 'SC5314_TP0_YPD_4', 
    'SC5314_TP3_YPD_1', 'SC5314_TP3_YPD_2', 'SC5314_TP3_YPD_3', 'SC5314_TP3_YPD_4', 
    'SC5314_TP3_Low_FLZ_1', 'SC5314_TP3_Low_FLZ_2', 'SC5314_TP3_Low_FLZ_3', 'SC5314_TP3_Low_FLZ_4', 
    'SC5314_TP3_High_FLZ_1', 'SC5314_TP3_High_FLZ_2', 'SC5314_TP3_High_FLZ_3', 'SC5314_TP3_High_FLZ_4'
]

freq_col_list = [
    'freq_SC5314_TP0_YPD_1', 'freq_SC5314_TP0_YPD_2', 'freq_SC5314_TP0_YPD_3', 'freq_SC5314_TP0_YPD_4', 
    'freq_SC5314_TP3_YPD_1', 'freq_SC5314_TP3_YPD_2', 'freq_SC5314_TP3_YPD_3', 'freq_SC5314_TP3_YPD_4', 
    'freq_SC5314_TP3_Low_FLZ_1', 'freq_SC5314_TP3_Low_FLZ_2', 'freq_SC5314_TP3_Low_FLZ_3', 'freq_SC5314_TP3_Low_FLZ_4', 
    'freq_SC5314_TP3_High_FLZ_1', 'freq_SC5314_TP3_High_FLZ_2', 'freq_SC5314_TP3_High_FLZ_3', 'freq_SC5314_TP3_High_FLZ_4'
]

The following block basically filters gRNAs that have a frequency that corresponds to them being below the 5th percentile, and also adds a "dynamic epsilon" value to the threshold to ensure that each gRNA has at least a frequency above that which would be present if the count was "1" in a given sample. This is necessary since there are a lot of gRNAs that drop out (and have 0 counts), so if you just filtered with the 5th percentile it wouldn't actually cut anything.

In [8]:
#Define the groups based on your frequency columns
groups = {
    "TP0_YPD": ["freq_SC5314_TP0_YPD_1", "freq_SC5314_TP0_YPD_2", "freq_SC5314_TP0_YPD_3", "freq_SC5314_TP0_YPD_4"],
    "TP3_YPD": ["freq_SC5314_TP3_YPD_1", "freq_SC5314_TP3_YPD_2", "freq_SC5314_TP3_YPD_3", "freq_SC5314_TP3_YPD_4"],
    "TP3_Low_FLZ": ["freq_SC5314_TP3_Low_FLZ_1", "freq_SC5314_TP3_Low_FLZ_2", "freq_SC5314_TP3_Low_FLZ_3", "freq_SC5314_TP3_Low_FLZ_4"],
    "TP3_High_FLZ": ["freq_SC5314_TP3_High_FLZ_1", "freq_SC5314_TP3_High_FLZ_2", "freq_SC5314_TP3_High_FLZ_3", "freq_SC5314_TP3_High_FLZ_4"]
}

#We have 4 replicates for each sample here, so we can make it so that the sgRNA only needs to pass the threshold in 3 replicates in every group (this is still usable)
min_replicates = 3
valid_gRNAs = set(frequencies_df.index)
threshold_values = {}

#Compute total counts per sample (sum of original counts)
total_counts = frequencies_df[count_col_list].sum(axis=0)

for group, samples in groups.items():
    group_df = frequencies_df[samples].copy()
    
    #Compute per-sample thresholds: 5th percentile + epsilon
    thresholds = {}
    for sample in samples:
        #5th percentile of non-zero frequencies
        percentile_5 = np.percentile(group_df[sample][group_df[sample] > 0], 5) if any(group_df[sample] > 0) else 0
        #Dynamic epsilon: frequency that a single count contributes
        epsilon = 1 / total_counts[sample.replace("freq_", "")]
        thresholds[sample] = percentile_5 + epsilon
        threshold_values[sample] = thresholds[sample]
    
    #Compare each row to the per-sample thresholds
    valid_gRNAs_group = group_df.ge(pd.Series(thresholds)).sum(axis=1) >= min_replicates
    valid_gRNAs &= set(valid_gRNAs_group[valid_gRNAs_group].index)

#Metadata columns to keep
metadata_cols = ["Alias", "Gene", "Barcode", "Position"]

#Combine metadata, counts, and frequency columns for filtered gRNAs
filtered_df = frequencies_df.loc[list(valid_gRNAs), metadata_cols + count_col_list + freq_col_list]

#Save to CSV
filtered_df.to_csv("filtered_gRNAs.csv", index=False)

#Optional: print the threshold
common_threshold = min(threshold_values.values())
print("Common Frequency Threshold:", common_threshold)

Common Frequency Threshold: 1.6192764049423182e-07


In [9]:
filtered_df

,Alias,Gene,Barcode,Position,SC5314_TP0_YPD_1,SC5314_TP0_YPD_2,SC5314_TP0_YPD_3,SC5314_TP0_YPD_4,SC5314_TP3_YPD_1,SC5314_TP3_YPD_2,...,freq_SC5314_TP3_YPD_3,freq_SC5314_TP3_YPD_4,freq_SC5314_TP3_Low_FLZ_1,freq_SC5314_TP3_Low_FLZ_2,freq_SC5314_TP3_Low_FLZ_3,freq_SC5314_TP3_Low_FLZ_4,freq_SC5314_TP3_High_FLZ_1,freq_SC5314_TP3_High_FLZ_2,freq_SC5314_TP3_High_FLZ_3,freq_SC5314_TP3_High_FLZ_4
Number,,,,,,,,,,,,,,,,,,,,,
2,CR07390CA_276_revcom,CR07390C_A,GAGTATAGTGATCCATGTGC,1605740.0,2017.0,1006.0,956.0,2949.0,344.0,811.0,...,0.000038,0.000040,0.000031,0.000071,0.000145,0.000018,0.000025,4.704087e-05,8.991621e-08,0.000087
3,CR07390CA_239_revcom,CR07390C_A,GCTATAACGTTACTAGTAGT,1605777.0,88867.0,86080.0,83323.0,116333.0,94225.0,81253.0,...,0.008945,0.009431,0.007988,0.007907,0.008200,0.008257,0.007892,7.885633e-03,6.512092e-03,0.007555
5,CR07390CA_212_revcom,CR07390C_A,TCCATCATCATTATTACAAT,1605804.0,1182.0,862.0,952.0,1604.0,1415.0,943.0,...,0.000117,0.000153,0.000083,0.000091,0.000096,0.000113,0.000241,8.228270e-05,1.572635e-04,0.000091
6,CR07390CA_70,CR07390C_A,CTCTTTTGCTCTCATTTTTA,1605924.0,1006.0,1401.0,1421.0,1415.0,1658.0,2027.0,...,0.000216,0.000163,0.000149,0.000162,0.000213,0.000149,0.000071,7.219143e-05,5.493881e-05,0.000149
8,CR10440WA_257,CR10440W_A,AAGAATATTGTGTTGGTTCC,2222738.0,905.0,765.0,806.0,1251.0,1209.0,462.0,...,0.000064,0.000122,0.000053,0.000010,0.000048,0.000108,0.000128,1.552504e-07,4.801526e-05,0.000119
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5695,non-targeting55,non-targeting55,AGTGTCCCGTCATGCTCCAG,NaN,2536.0,3181.0,2740.0,4228.0,3344.0,3844.0,...,0.000325,0.000333,0.000429,0.000489,0.000425,0.000377,0.000463,3.376696e-04,1.908022e-04,0.000383
5696,non-targeting56,non-targeting56,TCTTTCGGGAGCGCCTGATT,NaN,3300.0,3063.0,2857.0,4753.0,2746.0,2848.0,...,0.000261,0.000277,0.000231,0.000277,0.000226,0.000430,0.000445,4.797237e-05,3.497741e-04,0.000319
5697,non-targeting57,non-targeting57,TCCCCAAGGCTGTCCCCCAA,NaN,1437.0,926.0,1189.0,1853.0,1421.0,672.0,...,0.000077,0.000143,0.000152,0.000090,0.000027,0.000138,0.000067,9.330548e-05,4.954383e-05,0.000108


## Calculate Log2 Fold-Changes w/ Statistics

In [10]:
#Make a copy so we don't touch the original
statistics_df = filtered_df.copy()

def benjamini_hochberg(p_values):
    
    #Compute BH FDR-adjusted p-values.
    p_values = np.array(p_values)
    n = len(p_values)
    sorted_indices = np.argsort(p_values)
    sorted_p = p_values[sorted_indices]
    adjusted = np.empty(n, dtype=float)
    cummin = 1.0
    for i in reversed(range(n)):
        rank = i + 1
        fdr = sorted_p[i] * n / rank
        cummin = min(cummin, fdr)
        adjusted[sorted_indices[i]] = cummin
    return np.minimum(adjusted, 1.0)

#Initialize lists
log2_fc_ypd, log2_fc_low_flz, log2_fc_high_flz = [], [], []
p_ypd, p_low_flz, p_high_flz = [], [], []

for gRNA in statistics_df.index:
    
    #Extract replicate frequencies
    tp0 = statistics_df.loc[gRNA, ['freq_SC5314_TP0_YPD_1','freq_SC5314_TP0_YPD_2','freq_SC5314_TP0_YPD_3','freq_SC5314_TP0_YPD_4']]
    tp3_ypd = statistics_df.loc[gRNA, ['freq_SC5314_TP3_YPD_1','freq_SC5314_TP3_YPD_2','freq_SC5314_TP3_YPD_3','freq_SC5314_TP3_YPD_4']]
    tp3_low_flz = statistics_df.loc[gRNA, ['freq_SC5314_TP3_Low_FLZ_1','freq_SC5314_TP3_Low_FLZ_2','freq_SC5314_TP3_Low_FLZ_3','freq_SC5314_TP3_Low_FLZ_4']]
    tp3_high_flz = statistics_df.loc[gRNA, ['freq_SC5314_TP3_High_FLZ_1','freq_SC5314_TP3_High_FLZ_2','freq_SC5314_TP3_High_FLZ_3','freq_SC5314_TP3_High_FLZ_4']]

    #Filter replicates below threshold and convert to float (ie. even if a sgRNA passed from earlier, we don't want to include a replicate of that sgRNA in our calculations that did not meet the threhold)
    tp0_filtered = tp0[tp0 >= [threshold_values[s] for s in tp0.index]].dropna().astype(float)
    tp3_ypd_filtered = tp3_ypd[tp3_ypd >= [threshold_values[s] for s in tp3_ypd.index]].dropna().astype(float)
    tp3_low_flz_filtered = tp3_low_flz[tp3_low_flz >= [threshold_values[s] for s in tp3_low_flz.index]].dropna().astype(float)
    tp3_high_flz_filtered = tp3_high_flz[tp3_high_flz >= [threshold_values[s] for s in tp3_high_flz.index]].dropna().astype(float)

    #Skip gRNAs with too few replicates
    if len(tp0_filtered) < 3 or len(tp3_ypd_filtered) < 3 or len(tp3_low_flz_filtered) < 3 or len(tp3_high_flz_filtered) < 3:
        log2_fc_ypd.append(np.nan)
        log2_fc_low_flz.append(np.nan)
        log2_fc_high_flz.append(np.nan)
        p_ypd.append(np.nan)
        p_low_flz.append(np.nan)
        p_high_flz.append(np.nan)
        continue

    #Compute log2 fold changes
    tp0_mean = np.mean(tp0_filtered)
    log2_fc_ypd.append(np.log2(np.mean(tp3_ypd_filtered)/tp0_mean) if tp0_mean > 0 else np.nan)
    log2_fc_low_flz.append(np.log2(np.mean(tp3_low_flz_filtered)/tp0_mean) if tp0_mean > 0 else np.nan)
    log2_fc_high_flz.append(np.log2(np.mean(tp3_high_flz_filtered)/tp0_mean) if tp0_mean > 0 else np.nan)

    #Compute p-values
    p_ypd.append(stats.ttest_ind(tp3_ypd_filtered, tp0_filtered, nan_policy='omit')[1])
    p_low_flz.append(stats.ttest_ind(tp3_low_flz_filtered, tp0_filtered, nan_policy='omit')[1])
    p_high_flz.append(stats.ttest_ind(tp3_high_flz_filtered, tp0_filtered, nan_policy='omit')[1])

#Adjust p-values using BH FDR
adj_p_ypd = benjamini_hochberg(p_ypd)
adj_p_low_flz = benjamini_hochberg(p_low_flz)
adj_p_high_flz = benjamini_hochberg(p_high_flz)

#Add results to statistics_df with original column names for log2 fold changes
statistics_df['log2_fold_change_ypd'] = log2_fc_ypd
statistics_df['log2_fold_change_low_flz'] = log2_fc_low_flz
statistics_df['log2_fold_change_high_flz'] = log2_fc_high_flz
statistics_df['p_value_ypd'] = p_ypd
statistics_df['p_value_low_flz'] = p_low_flz
statistics_df['p_value_high_flz'] = p_high_flz
statistics_df['adj_p_value_ypd'] = adj_p_ypd
statistics_df['adj_p_value_low_flz'] = adj_p_low_flz
statistics_df['adj_p_value_high_flz'] = adj_p_high_flz

#Save to CSV
statistics_df.to_csv("statistics_df_with_log2fc_and_fdr.csv")